# Cross-Dataset Validation: HCP Models vs TMS-fMRI Empirical Data

**Objective**: Test whether HCP-trained ANN models can generalize to TMS-fMRI data by comparing mean FC (rest, stim, delta) grouped by stimulation target.

**Methodology**:
- For each stimulation target in the TMS dataset:
  - Generate 1 simulation session per HCP subject (all subjects)
  - Average FC matrices across HCP subjects → **HCP mean FC**
  - Average FC matrices across TMS subjects at that target → **TMS empirical mean FC**
  - Compare HCP mean and TMS empirical mean via Pearson correlation

**Output**: Per-target validation metrics (r_rest, r_stim, r_delta)

## Step 1: Setup and Configuration

In [ ]:
# Setup Google Colab (if needed)
import sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = '/content/drive/My Drive/BrainStim_ANN_fMRI_HCP'
except ImportError:
    base_dir = '/Users/cbc/Documents/GitHub/fufo/notebook/ANN_background/BrainStim_ANN_fMRI_HCP-main'

sys.path.insert(0, base_dir)

import numpy as np
import pandas as pd
import torch
import pickle
from scipy.signal import butter, filtfilt
from scipy.spatial.distance import cdist
from collections import defaultdict

print(f"Working directory: {base_dir}")

## Step 2: Configuration Parameters

In [ ]:
# Data directories
data_dir = base_dir
hcp_weights_dir = f'{data_dir}/trained_models/HCP_models'
tms_data_pkl = f'{data_dir}/data/TMS_fMRI/tms_preprocessed.pkl'
dist_matrix_path = f'{data_dir}/data/Schaefer400_Tian50_distance.npy'

# ROI configuration
cortical_roi_indices = np.arange(50, 450)  # Schaefer 400 (exclude Tian 50 subcortical)
n_rois = len(cortical_roi_indices)  # 400 cortical ROIs

# Simulation parameters
tr_hcp = 0.72  # HCP effective TR for stimulus timing precision
tr_stim = 2.4  # TMS fMRI TR (from empirical data)
ds_factor = tr_stim / tr_hcp  # ~3.33x downsampling
using_steps = 3  # Rolling window size for ANN input
burn_in = 30  # Steps before stimulus application
stim_amp = 10.0  # Stimulation amplitude
noise_sigma = 0.28  # Stochastic noise

print(f"Configuration:")
print(f"  Cortical ROIs: {n_rois}")
print(f"  TR HCP (sim): {tr_hcp}s, TR TMS (emp): {tr_stim}s")
print(f"  Downsampling factor: {ds_factor:.2f}x")
print(f"  Burn-in: {burn_in} steps")

## Step 3: Load Distance Matrix

In [ ]:
# Load distance matrix and create spatial Gaussian kernel
D = np.load(dist_matrix_path)  # Shape: (450, 450)
print(f"Distance matrix shape: {D.shape}")

# Create spatial Gaussian kernel W
sigma_w = 100.0  # Spatial smoothing parameter
W = np.exp(-(D ** 2) / (2 * sigma_w ** 2))
W = W / np.sum(W, axis=1, keepdims=True)  # Row-normalize
print(f"Spatial kernel W shape: {W.shape}")

## Step 4: Load HCP Models

In [ ]:
# Load all trained HCP models
import os
trained_models = {}
rest_data_hcp = {}

for model_file in sorted(os.listdir(hcp_weights_dir)):
    if model_file.endswith('.pkl'):
        model_path = os.path.join(hcp_weights_dir, model_file)
        with open(model_path, 'rb') as f:
            model_dict = pickle.load(f)
            sub_id = model_dict['subject_id']
            trained_models[sub_id] = model_dict['model']
            rest_data_hcp[sub_id] = model_dict.get('rest_data', None)
            
print(f"Loaded {len(trained_models)} HCP models:")
for sub_id in sorted(trained_models.keys()):
    print(f"  {sub_id}")

## Step 5: Load Empirical TMS Data and Group by Target

In [ ]:
# Load empirical TMS-fMRI dataset
with open(tms_data_pkl, 'rb') as f:
    dataset_emp = pickle.load(f)

print(f"TMS empirical dataset: {len(dataset_emp)} subjects")

# Group empirical TMS data by target
from src.preprocessing_tms_fmri import get_tms_target_list

empirical_by_target = defaultdict(list)  # target_idx → [{sub_id, rest_fc, stim_fc, delta_fc, n_sessions}, ...]

for sub_id in sorted(dataset_emp.keys()):
    subject_data = dataset_emp[sub_id]
    
    # Get rest sessions and compute mean REST FC
    rest_sessions = subject_data.get('rest', [])
    if len(rest_sessions) > 0:
        rest_ts_list = [session['ts'] for session in rest_sessions if 'ts' in session]
        if len(rest_ts_list) > 0:
            # Concatenate rest sessions and compute FC
            rest_ts_concat = np.concatenate(rest_ts_list, axis=0)  # Shape: (T_total, 450)
            rest_fc = compute_fc_matrix(rest_ts_concat[:, cortical_roi_indices])
        else:
            continue
    else:
        continue
    
    # Get stim sessions and group by target
    stim_sessions = subject_data.get('stim', [])
    for stim_session in stim_sessions:
        if 'ts' not in stim_session or 'target_idx' not in stim_session:
            continue
        
        target_idx = stim_session['target_idx']
        stim_ts = stim_session['ts']
        
        # Compute stim FC and delta FC
        stim_fc = compute_fc_matrix(stim_ts[:, cortical_roi_indices])
        delta_fc = stim_fc - rest_fc
        
        empirical_by_target[target_idx].append({
            'sub_id': sub_id,
            'rest_fc': rest_fc,
            'stim_fc': stim_fc,
            'delta_fc': delta_fc
        })

print(f"\nEmpirical data grouped by target:")
for target_idx in sorted(empirical_by_target.keys()):
    n_entries = len(empirical_by_target[target_idx])
    print(f"  Target {target_idx}: {n_entries} sessions")

## Step 6: Helper Functions for Simulation

In [ ]:
def safe_target_idx(target_label, roi_count=450):
    """Convert target label to safe ROI index."""
    if isinstance(target_label, str):
        try:
            idx = int(target_label.split('_')[-1])
        except:
            idx = 0
    else:
        idx = int(target_label)
    return np.clip(idx, 0, roi_count - 1)

def get_onset_column(df, preferred_cols=['onset', 'OnsetTime', 'stim_time']):
    """Find stimulus onset column in dataframe."""
    for col in preferred_cols:
        if col in df.columns:
            return col
    return df.columns[0] if len(df.columns) > 0 else None

def map_onsets_to_steps(onsets_seconds, tr_model=0.72):
    """Map onset times (in seconds) to simulation steps at high resolution."""
    return np.round(onsets_seconds / tr_model).astype(int)

print("Helper functions defined.")

## Step 7: ANN Inference Functions

In [ ]:
def predict_next(model, x, target_idx, W, device='cpu'):
    """
    Predict next timestep using ANN model.
    
    Args:
        model: PyTorch ANN model
        x: Input activity (using_steps, 450)
        target_idx: Target ROI index
        W: Spatial kernel (450, 450)
    
    Returns:
        next_step: Predicted activity (450,)
    """
    with torch.no_grad():
        # Prepare input
        x_input = torch.tensor(x.flatten(), dtype=torch.float32, device=device).unsqueeze(0)
        
        # Forward pass
        x_pred = model(x_input).cpu().numpy()[0]
    
    return x_pred

def simulate_run(model, init_state, n_steps, stim_steps=None, target_idx=0, W=None, 
                 burn_in=30, stim_amp=10.0, noise_sigma=0.28, device='cpu'):
    """
    Simulate ANN for n_steps with optional stimulation.
    
    Args:
        model: PyTorch ANN model
        init_state: Initial activity (using_steps, 450)
        n_steps: Number of simulation steps
        stim_steps: Array of step indices where to apply stimulation
        target_idx: Target ROI for stimulation
        W: Spatial kernel
        burn_in: Steps before applying stim
        stim_amp: Stimulation amplitude
        noise_sigma: Noise standard deviation
    
    Returns:
        ts: Full timeseries (n_steps, 450)
    """
    if stim_steps is None:
        stim_steps = []
    
    ts = np.zeros((n_steps, 450))
    ts[:using_steps] = init_state.copy()
    
    x = init_state.copy()
    
    for t in range(using_steps, n_steps):
        # Predict next step
        x_next = predict_next(model, x, target_idx, W, device=device)
        
        # Add noise
        noise = np.random.normal(0, noise_sigma, size=x_next.shape)
        x_next = x_next + noise
        
        # Apply stimulation after burn-in
        if t > burn_in and t in stim_steps:
            x_next[target_idx] += stim_amp
        
        # Update rolling window
        x = np.vstack([x[1:], x_next])
        
        # Store full step
        ts[t] = x_next
    
    return ts

print("Inference functions defined.")

## Step 8: Functional Connectivity Computation

In [ ]:
def compute_fc_matrix(timeseries):
    """
    Compute Fisher-z transformed FC matrix from timeseries.
    
    Args:
        timeseries: (T, n_rois) array
    
    Returns:
        fc_z: Fisher-z transformed FC (n_rois, n_rois)
    """
    # Compute Pearson correlation
    fc = np.corrcoef(timeseries.T)
    
    # Fisher-z transform
    fc_z = 0.5 * np.log((1 + fc) / (1 - fc + 1e-6))
    fc_z = np.nan_to_num(fc_z, nan=0.0, posinf=0.0, neginf=0.0)
    
    return fc_z

def extract_upper_triangle(fc_matrix, k=1):
    """
    Extract upper triangle (excluding diagonal) of symmetric matrix.
    
    Args:
        fc_matrix: (n_rois, n_rois) symmetric matrix
        k: Diagonal offset (k=1 excludes main diagonal)
    
    Returns:
        vec: Flattened upper triangle
    """
    indices = np.triu_indices(fc_matrix.shape[0], k=k)
    return fc_matrix[indices]

def downsample_timeseries(ts_hires, ds_factor):
    """
    Downsample timeseries by averaging blocks.
    
    Args:
        ts_hires: (T, n_rois) high-resolution timeseries
        ds_factor: Downsampling factor
    
    Returns:
        ts_downsampled: (T_down, n_rois) downsampled timeseries
    """
    ds_factor = int(np.round(ds_factor))
    T = ts_hires.shape[0]
    T_down = T // ds_factor
    ts_downsampled = ts_hires[:T_down * ds_factor].reshape(T_down, ds_factor, -1).mean(axis=1)
    return ts_downsampled

print("FC computation functions defined.")

## Step 9: Simulate One Session per HCP Subject per Target

In [ ]:
# Storage for simulated data: target_idx → [{hcp_sub_id, rest_fc, stim_fc, delta_fc}, ...]
simulated_by_target = defaultdict(list)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# For each target in empirical data
for target_idx in sorted(empirical_by_target.keys()):
    print(f"\n=== Target {target_idx} ===")
    
    # Get one empirical stim run at this target to get onsets and duration
    emp_sample = empirical_by_target[target_idx][0]  # First entry has empirical stim data
    
    # Get empirical stim onsets for this target from TMS dataset
    # Extract from first TMS subject at this target
    tms_sub_id = emp_sample['sub_id']
    stim_df = dataset_emp[tms_sub_id]['stim_onsets_df']  # Stimulus timing dataframe
    
    # Filter for this target
    target_stim_df = stim_df[stim_df['target_idx'] == target_idx]
    if len(target_stim_df) == 0:
        print(f"  No stimulus data for target {target_idx}")
        continue
    
    # Get onset column and extract onsets
    onset_col = get_onset_column(target_stim_df)
    onsets_s = target_stim_df[onset_col].values  # In seconds
    
    # Get stim duration
    stim_duration_s = target_stim_df.get('duration', pd.Series([0.5])).iloc[0]  # Default 0.5s
    
    # Get TMS session duration (from empirical stim timeseries)
    emp_stim_ts = emp_sample['ts']  # This comes from empirical stim session
    n_tms_steps = emp_stim_ts.shape[0]
    tms_duration_s = n_tms_steps * tr_stim
    
    # Calculate HCP simulation length (at high resolution)
    n_hcp_steps = int(np.ceil(n_tms_steps * ds_factor))
    
    # Map stimulus onsets to HCP high-resolution steps
    stim_steps_hires = map_onsets_to_steps(onsets_s, tr_model=tr_hcp)
    
    print(f"  Empirical: {n_tms_steps} steps @ {tr_stim}s = {tms_duration_s:.1f}s")
    print(f"  HCP sim: {n_hcp_steps} steps @ {tr_hcp}s = {n_hcp_steps*tr_hcp:.1f}s")
    print(f"  Stimulus onsets (HCP steps): {stim_steps_hires}")
    print(f"  Processing {len(trained_models)} HCP subjects...")
    
    # For each HCP subject model
    for hcp_sub_id in sorted(trained_models.keys()):
        model = trained_models[hcp_sub_id]
        
        # Get initialization from HCP rest data
        if hcp_sub_id in rest_data_hcp and rest_data_hcp[hcp_sub_id] is not None:
            rest_ts = rest_data_hcp[hcp_sub_id]
        else:
            print(f"    Warning: No rest data for HCP subject {hcp_sub_id}, skipping")
            continue
        
        # Extract first 3 timepoints as initialization
        if len(rest_ts) < using_steps:
            print(f"    Warning: Rest data too short for HCP subject {hcp_sub_id}")
            continue
        
        rest_init_window = rest_ts[:using_steps].copy()  # (using_steps, 450)
        
        # Compute HCP rest FC
        rest_fc_hcp = compute_fc_matrix(rest_ts[:, cortical_roi_indices])
        
        # Simulate at high resolution with stimulus
        try:
            sim_ts_hires = simulate_run(
                model,
                rest_init_window,
                n_hcp_steps,
                stim_steps=stim_steps_hires,
                target_idx=target_idx,
                W=W,
                burn_in=burn_in,
                stim_amp=stim_amp,
                noise_sigma=noise_sigma,
                device=device
            )  # Shape: (n_hcp_steps, 450)
        except Exception as e:
            print(f"    Error simulating {hcp_sub_id}: {str(e)}")
            continue
        
        # Downsample to TMS TR
        sim_ts_downsampled = downsample_timeseries(sim_ts_hires, ds_factor)[:n_tms_steps]
        
        # Compute stim FC and delta FC
        stim_fc_hcp = compute_fc_matrix(sim_ts_downsampled[:, cortical_roi_indices])
        delta_fc_hcp = stim_fc_hcp - rest_fc_hcp
        
        # Store
        simulated_by_target[target_idx].append({
            'hcp_sub_id': hcp_sub_id,
            'rest_fc': rest_fc_hcp,
            'stim_fc': stim_fc_hcp,
            'delta_fc': delta_fc_hcp
        })
    
    print(f"  Simulated: {len(simulated_by_target[target_idx])} HCP subjects")

print(f"\n=== Simulation Complete ===")
print(f"Targets with simulations: {sorted(simulated_by_target.keys())}")

## Step 10: Target-Averaged Validation Comparison

In [ ]:
# Storage for validation results
validation_results = {}

print("\n=== TARGET-BASED VALIDATION ===")
print(f"{'Target':<8} {'r_rest':>8} {'r_stim':>8} {'r_delta':>8} {'N_HCP':>6} {'N_TMS':>6}")
print("-" * 48)

for target_idx in sorted(empirical_by_target.keys()):
    if target_idx not in simulated_by_target:
        continue
    
    # Get empirical entries
    emp_entries = empirical_by_target[target_idx]
    sim_entries = simulated_by_target[target_idx]
    
    n_emp = len(emp_entries)
    n_sim = len(sim_entries)
    
    # Average FC across empirical subjects
    emp_rest_fc_mean = np.mean([e['rest_fc'] for e in emp_entries], axis=0)
    emp_stim_fc_mean = np.mean([e['stim_fc'] for e in emp_entries], axis=0)
    emp_delta_fc_mean = np.mean([e['delta_fc'] for e in emp_entries], axis=0)
    
    # Average FC across simulated HCP subjects
    sim_rest_fc_mean = np.mean([s['rest_fc'] for s in sim_entries], axis=0)
    sim_stim_fc_mean = np.mean([s['stim_fc'] for s in sim_entries], axis=0)
    sim_delta_fc_mean = np.mean([s['delta_fc'] for s in sim_entries], axis=0)
    
    # Extract upper triangles
    emp_rest_vec = extract_upper_triangle(emp_rest_fc_mean)
    sim_rest_vec = extract_upper_triangle(sim_rest_fc_mean)
    
    emp_stim_vec = extract_upper_triangle(emp_stim_fc_mean)
    sim_stim_vec = extract_upper_triangle(sim_stim_fc_mean)
    
    emp_delta_vec = extract_upper_triangle(emp_delta_fc_mean)
    sim_delta_vec = extract_upper_triangle(sim_delta_fc_mean)
    
    # Compute correlations
    r_rest = np.corrcoef(emp_rest_vec, sim_rest_vec)[0, 1]
    r_stim = np.corrcoef(emp_stim_vec, sim_stim_vec)[0, 1]
    r_delta = np.corrcoef(emp_delta_vec, sim_delta_vec)[0, 1]
    
    # Handle NaN
    r_rest = r_rest if not np.isnan(r_rest) else 0.0
    r_stim = r_stim if not np.isnan(r_stim) else 0.0
    r_delta = r_delta if not np.isnan(r_delta) else 0.0
    
    # Store results
    validation_results[target_idx] = {
        'r_rest': r_rest,
        'r_stim': r_stim,
        'r_delta': r_delta,
        'n_hcp': n_sim,
        'n_tms': n_emp
    }
    
    # Print
    print(f"{target_idx:<8} {r_rest:>8.4f} {r_stim:>8.4f} {r_delta:>8.4f} {n_sim:>6} {n_emp:>6}")

print("-" * 48)

## Step 11: Summary Statistics

In [ ]:
# Compile results into DataFrame for easier viewing
results_df = pd.DataFrame([
    {
        'target': target_idx,
        'r_rest': validation_results[target_idx]['r_rest'],
        'r_stim': validation_results[target_idx]['r_stim'],
        'r_delta': validation_results[target_idx]['r_delta'],
        'n_hcp': validation_results[target_idx]['n_hcp'],
        'n_tms': validation_results[target_idx]['n_tms']
    }
    for target_idx in sorted(validation_results.keys())
])

# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"\nNumber of targets validated: {len(results_df)}")
print(f"\nREST FC Correlation (r_rest):")
print(f"  Mean: {results_df['r_rest'].mean():.4f}")
print(f"  Std:  {results_df['r_rest'].std():.4f}")
print(f"  Min:  {results_df['r_rest'].min():.4f}")
print(f"  Max:  {results_df['r_rest'].max():.4f}")

print(f"\nSTIM FC Correlation (r_stim):")
print(f"  Mean: {results_df['r_stim'].mean():.4f}")
print(f"  Std:  {results_df['r_stim'].std():.4f}")
print(f"  Min:  {results_df['r_stim'].min():.4f}")
print(f"  Max:  {results_df['r_stim'].max():.4f}")

print(f"\nDelta FC Correlation (r_delta):")
print(f"  Mean: {results_df['r_delta'].mean():.4f}")
print(f"  Std:  {results_df['r_delta'].std():.4f}")
print(f"  Min:  {results_df['r_delta'].min():.4f}")
print(f"  Max:  {results_df['r_delta'].max():.4f}")

print(f"\n=== INTERPRETATION ===")
r_delta_mean = results_df['r_delta'].mean()
r_stim_mean = results_df['r_stim'].mean()
print(f"Delta > Stim: {r_delta_mean > r_stim_mean}")
if r_delta_mean > r_stim_mean:
    print("  ✓ Model captures stimulus-induced FC changes (delta > stim)")
else:
    print("  ✗ Model does not capture stimulus-induced FC specificity (delta ≤ stim)")

print(f"\nCross-dataset generalization: Mean r_stim = {r_stim_mean:.4f}")
if r_stim_mean > 0.3:
    print("  ✓ Good generalization (r > 0.3)")
elif r_stim_mean > 0.1:
    print("  ~ Moderate generalization (0.1 < r < 0.3)")
else:
    print("  ✗ Poor generalization (r < 0.1)")

## Results Table

In [ ]:
# Display full results table
print("\nDetailed Results by Target:")
print(results_df.to_string(index=False))

# Save results
results_pkl = f'{data_dir}/results/HCP_to_TMS_validation_results.pkl'
with open(results_pkl, 'wb') as f:
    pickle.dump({
        'results_df': results_df,
        'validation_results': validation_results,
        'empirical_by_target': dict(empirical_by_target),
        'simulated_by_target': dict(simulated_by_target)
    }, f)
print(f"\nResults saved to: {results_pkl}")